<a href="https://colab.research.google.com/github/conce1994/AI-Customer-Support-Agent/blob/main/AI_Customer_Support_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **AI Customer Support Agent**

Step 1. Install the Required Libraries:

Google Colab starts with a clean environment every time you open it. That means it doesn't know about libraries like LangChain or ChromaDB until we install them.

*We're installing:*

-openai:	Communicates with OpenAI models.

-langchain:	Provides tools to build LLM applications.

-langchain-openai:	Connects LangChain to OpenAI models.

-chromadb:	Stores and searches document embeddings (our knowledge base).

-tiktoken:	Counts tokens so text can be split efficiently.

-python-dotenv:	Loads API keys from environment variables (useful outside Colab).

In [168]:
!pip install -q --upgrade openai langchain langchain-openai chromadb tiktoken python-dotenv langchain-text-splitters sentence-transformers google-generativeai langchain-community

Step 2: Import Libraries:

***Document:*** This represents a document that the AI can search.Instead of storing plain text, LangChain stores objects like:

Document(
    page_content="Refunds are available within 30 days."
)

***RecursiveCharacterTextSplitter:***

LLMs have context limits.

Imagine giving a model a 300-page employee handbook.

It cannot read the entire document every time someone asks:

"How many vacation days do employees receive?"

Instead, we split documents into manageable chunks.

Example:

Employee Handbook

↓

Chunk 1, Chunk 2, Chunk 3, Chunk 4, ...

The AI searches only the relevant chunks instead of reading the whole document.

This is one of the key ideas behind Retrieval-Augmented Generation (RAG).


**OpenAIEmbeddings**:

Computers don't understand text the way humans do.

We convert text into numbers called embeddings.


"What is your refund policy?"

↓

[0.12, -0.44, 0.87, ...]

A sentence with a similar meaning will produce a similar vector.

Refund policy

↓

[0.13, -0.41, 0.89, ...]

Because these vectors are close together, the system knows they're related.

This enables semantic search rather than simple keyword matching.

***ChatOpenAI***:

This is the Large Language Model that will generate answers.

Think of the architecture like this:

1.User Question

2.Retrieve Relevant Documents

3.Send Question + Documents
      
4.ChatOpenAI

5.Final Answer

The model doesn't search the documents itself. It relies on the retriever to provide the most relevant information.

In [214]:
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import google.generativeai as genai
from google.colab import userdata

Step 3: Configure Your OpenAI API Key

The OpenAI API requires authentication before it can process requests.

Instead of hardcoding the key:

we securely enter it when the notebook runs.

Why this is important

Never upload API keys to GitHub.

Instead, professional projects use:

Environment variables
.env files (for local development)
Secret managers (for cloud deployments)

This is an essential security practice you'll use in production systems.

In [215]:
import os
from getpass import getpass
from google.colab import userdata

# Securely set OpenAI API Key for ChatOpenAI (if still needed, or remove if switching entirely)
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key (or leave blank if not using): ")

# Configure Google Generative AI
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY') # Assumes you've saved your Gemini API key in Colab secrets
genai.configure(api_key=GOOGLE_API_KEY)

Enter your OpenAI API Key (or leave blank if not using): ··········


Step 4: Add sample company knowledge

This is our mini “company knowledge base.” In a real company, this would be replaced with help docs, PDFs, website pages, tickets, or internal policies

In [216]:
company_faq = """
Q: What is your refund policy?
A: Customers can request a refund within 30 days of purchase if the product has not been used extensively.

Q: How can I contact support?
A: Customers can contact support by emailing support@example.com.

Q: Do you offer enterprise plans?
A: Yes, enterprise plans include priority support, custom integrations, and dedicated account management.

Q: How long does onboarding take?
A: Standard onboarding takes 5 business days. Enterprise onboarding may take 2 to 4 weeks depending on integrations.

Q: Do you support single sign-on?
A: Yes, single sign-on is available for enterprise customers using SAML or OAuth.
"""

Step 5: Turn Text into documents:

This converts each FAQ block into a searchable Document.

Each document has:

page_content = the actual text
metadata = information about where it came from

In [217]:
from langchain_core.documents import Document

docs = [
    Document(page_content=section.strip(), metadata={"source": "company_faq"})
    for section in company_faq.split("\n\n")
    if section.strip()
]

docs

[Document(metadata={'source': 'company_faq'}, page_content='Q: What is your refund policy?\nA: Customers can request a refund within 30 days of purchase if the product has not been used extensively.'),
 Document(metadata={'source': 'company_faq'}, page_content='Q: How can I contact support?\nA: Customers can contact support by emailing support@example.com.'),
 Document(metadata={'source': 'company_faq'}, page_content='Q: Do you offer enterprise plans?\nA: Yes, enterprise plans include priority support, custom integrations, and dedicated account management.'),
 Document(metadata={'source': 'company_faq'}, page_content='Q: How long does onboarding take?\nA: Standard onboarding takes 5 business days. Enterprise onboarding may take 2 to 4 weeks depending on integrations.'),
 Document(metadata={'source': 'company_faq'}, page_content='Q: Do you support single sign-on?\nA: Yes, single sign-on is available for enterprise customers using SAML or OAuth.')]

Step 6: Create embeddings and vector database:

What happens here: FAQ text → embeddings → Chroma vector database → retriever

In [218]:
!pip install -q langchain-chroma
from langchain_chroma import Chroma

# Use HuggingFaceEmbeddings for local embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # A good balance of performance and size

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Removed: Redundant LLM initialization. LLM will be initialized in cell yDURKdsRCisP.
# llm = genai.GenerativeModel('gemini-pro-latest')

print("Setup complete with HuggingFace embeddings and Gemini LLM (llm initialized in a later cell)!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete with HuggingFace embeddings and Gemini LLM (llm initialized in a later cell)!


Step 7: Test Retrieval:

This checks whether the system can find the right company knowledge before we generate an AI answer.

Expected result: it should return the FAQ about single sign-on and possibly enterprise plans.

In [219]:
question = "Can I log in using my company account??"

relevant_docs = retriever.invoke(question)

for doc in relevant_docs:
    print(doc.page_content)
    print("Source:", doc.metadata["source"])
    print("---")

Q: Do you support single sign-on?
A: Yes, single sign-on is available for enterprise customers using SAML or OAuth.
Source: company_faq
---
Q: Do you support single sign-on?
A: Yes, single sign-on is available for enterprise customers using SAML or OAuth.
Source: company_faq
---


Example:

Possible user questions:

-"Can I get my money back?"

-"How do refunds work?"

-"Can I return my purchase?"

-"What's your cancellation policy?"

Even though the wording is different, the embedding model recognizes that these questions are semantically related to the refund policy.

In [220]:
question = "what is your refund policy??"

relevant_docs = retriever.invoke(question)

for doc in relevant_docs:
    print(doc.page_content)
    print("Source:", doc.metadata["source"])
    print("---")

Q: What is your refund policy?
A: Customers can request a refund within 30 days of purchase if the product has not been used extensively.
Source: company_faq
---
Q: What is your refund policy?
A: Customers can request a refund within 30 days of purchase if the product has not been used extensively.
Source: company_faq
---


Step 8: Create the LLM:

This creates the AI model that will write the final support answer.

temperature=0 means the model should be more consistent and less creative. For customer support, we want reliable answers, not imaginative ones.


In [225]:
from google.generativeai.types import GenerationConfig
import time

llm = genai.GenerativeModel(
    'gemini-3.5-flash',
    generation_config=GenerationConfig(temperature=0)
)

def call_llm_with_backoff(prompt, max_retries=5, initial_delay=1):
    delay = initial_delay
    for i in range(max_retries):
        try:
            response = llm.generate_content(prompt)
            return response
        except Exception as e:
            # Check if it's a TooManyRequests error or a similar transient error
            if 'TooManyRequests' in str(e) or '50' in str(e):
                print(f"Rate limit hit or transient error. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
            else:
                # Re-raise other types of exceptions immediately
                raise e
    raise Exception(f"Failed to call LLM after {max_retries} retries.")

In [224]:
print('Listing available models:')
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Listing available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-r

Step 9: Build the support agent function:

This is the main "agent" logic

What happens:

User question
↓
Retriever finds relevant FAQ entries
↓
We place those FAQ entries into the prompt
↓
LLM answers only using that information

Important part in this instruction:

Answer using only the company information below. That help reduce allucinations


In [226]:
def support_agent(question, llm_model):
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f"""
You are a helpful customer support assistant.

Answer the user's question using only the company information below.

If the answer is not in the company information, say:
"I don't know based on the available company information."

Company information:
{context}

User question:
{question}

Answer:
"""

    response = llm_model.generate_content(prompt)

    return response.text

Step 10: Test the agent

Question: Can I gent my money back

Expected answer:

Customers can request a refund within 30 days of purchase if the product has not bee

This works even though the FAQ says “refund policy” and the user asked “Can I get my money back?”

In [227]:
support_agent("Can I get my money back?", llm)

'Yes, you can request a refund within 30 days of purchase if the product has not been used extensively.'

Step 11: Test an unknow question

In [228]:
support_agent("Do you offer phone support?", llm)

"I don't know based on the available company information."

Step 12: Return answer with sources: This version returns a dictionary instead of only a text answer.

That is closer to how real AI products work because APIs usually return structured data like:

In [229]:
def support_agent_with_sources(question):
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f"""
You are a helpful customer support assistant.

Answer the user's question using only the company information below.

If the answer is not in the company information, say:
"I don't know based on the available company information."

Company information:
{context}

User question:
{question}

Answer:
"""

    response = llm.generate_content(prompt)

    sources = [doc.metadata["source"] for doc in relevant_docs]

    return {
        "question": question,
        "answer": response.text,
        "sources": list(set(sources))
    }

Step 13: Test with Sources:

In [107]:
result = support_agent_with_sources("Can my company use single sign-on?")

result

{'question': 'Can my company use single sign-on?',
 'answer': 'Yes, single sign-on is available for enterprise customers using SAML or OAuth.',
 'sources': ['company_faq']}

Step 14: Test multiple user questions

we are testing three important behaviors:

1. Can it answer known questions?
2. Can it understand different wording?
3. Can it refuse unknown questions?


In [109]:

test_questions = [
    "Can I get a refund?",
    "How do I contact support?",
    "Do you have enterprise plans?",
    "Can employees log in with SSO?",
    "Do you offer phone support?",
    "How long does onboarding take?"
]

for question in test_questions:
    result = support_agent_with_sources(question)

    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Sources:", result["sources"])
    print("-" * 80)

Question: Can I get a refund?
Answer: You can request a refund within 30 days of purchase if the product has not been used extensively.
Sources: ['company_faq']
--------------------------------------------------------------------------------
Question: How do I contact support?
Answer: Customers can contact support by emailing support@example.com.
Sources: ['company_faq']
--------------------------------------------------------------------------------
Question: Do you have enterprise plans?
Answer: Yes, enterprise plans include priority support, custom integrations, and dedicated account management.
Sources: ['company_faq']
--------------------------------------------------------------------------------
Question: Can employees log in with SSO?
Answer: Yes, single sign-on is available for enterprise customers using SAML or OAuth.
Sources: ['company_faq']
--------------------------------------------------------------------------------
Question: Do you offer phone support?
Answer: I don't 

Step 15: Add a cleaner display function

This makes the output easier to read in Colab.

In [111]:
def ask_support_agent(question):
    result = support_agent_with_sources(question)

    print("Customer Question:")
    print(result["question"])
    print()

    print("AI Support Answer:")
    print(result["answer"])
    print()

    print("Sources Used:")
    for source in result["sources"]:
        print("-", source)

Step 16: Try the cleaner version

In [112]:
ask_support_agent("Can I use my company login?")

Customer Question:
Can I use my company login?

AI Support Answer:
Yes, single sign-on is available for enterprise customers using SAML or OAuth.

Sources Used:
- company_faq


Step 17: Create retriever with scores

Before, our retriever returned only documents.

Now it returns:

document + similarity score

The score tells us how close the question is to the retrieved company information

In [114]:
def retrieve_docs_with_scores(question, k=2):
    results = vectorstore.similarity_search_with_score(question, k=k)
    return results

Step 18: Inspect retrieval scores

This lets us see what the system retrieves when the user asks something that may not exist in the FAQ.

Lower scores usually mean a better match in Chroma/LangChain distance scoring.

We’ll use this to decide when the AI should say:

I don't know based on the available company information.

In [115]:
question = "Do you offer phone support?"

results = retrieve_docs_with_scores(question)

for doc, score in results:
    print("Score:", score)
    print(doc.page_content)
    print("-" * 80)

Score: 0.9476903676986694
Q: How can I contact support?
A: Customers can contact support by emailing support@example.com.
--------------------------------------------------------------------------------
Score: 0.9476903676986694
Q: How can I contact support?
A: Customers can contact support by emailing support@example.com.
--------------------------------------------------------------------------------


Step 19: Add a safer support agent:

This version does 4 things:

1. Searches the knowledge base
2. Checks the best similarity score
3. Refuses weak matches
4. Answers only when the retrieved info is strong enough

In [120]:
def safe_support_agent(question, threshold=0.8):
    results = retrieve_docs_with_scores(question, k=2)

    best_score = results[0][1]

    if best_score > threshold:
        return {
            "question": question,
            "answer": "I don't know based on the available company information.",
            "sources": [],
            "confidence": "low",
            "best_score": best_score
        }

    relevant_docs = [doc for doc, score in results]

    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f"""
You are a helpful customer support assistant.

Answer the user's question using only the company information below.

If the answer is not in the company information, say:
"I don't know based on the available company information."

Company information:
{context}

User question:
{question}

Answer:
"""

    response = llm.generate_content(prompt)

    sources = [doc.metadata["source"] for doc in relevant_docs]

    return {
        "question": question,
        "answer": response.text,
        "sources": list(set(sources)),
        "confidence": "high",
        "best_score": best_score
    }

Step 20: Test safe agent

This test both:

Known questions:
- refund
- SSO
- onboarding
- enterprise plans

Unknown questions:
- phone support
- Bitcoin payment

In [121]:
test_questions = [
    "Can I get my money back?",
    "Can I use my company login?",
    "How long does onboarding take?",
    "Do you offer phone support?",
    "Can I pay with Bitcoin?",
    "Do you have enterprise plans?"
]

for question in test_questions:
    result = safe_support_agent(question)

    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Confidence:", result["confidence"])
    print("Best score:", result["best_score"])
    print("Sources:", result["sources"])
    print("-" * 80)

Question: Can I get my money back?
Answer: I don't know based on the available company information.
Confidence: low
Best score: 1.1549369096755981
Sources: []
--------------------------------------------------------------------------------
Question: Can I use my company login?
Answer: I don't know based on the available company information.
Confidence: low
Best score: 1.080754280090332
Sources: []
--------------------------------------------------------------------------------
Question: How long does onboarding take?
Answer: Standard onboarding takes 5 business days. Enterprise onboarding may take 2 to 4 weeks depending on integrations.
Confidence: high
Best score: 0.35755661129951477
Sources: ['company_faq']
--------------------------------------------------------------------------------
Question: Do you offer phone support?
Answer: I don't know based on the available company information.
Confidence: low
Best score: 0.9476903676986694
Sources: []
--------------------------------------

Step 21: Create conversation history

In [230]:
conversation_history = []

Step 22: Add memory to the agent

In [231]:
def support_agent_with_better_memory(question, threshold=1.2):
    standalone_question = rewrite_question_with_history(question)

    results = retrieve_docs_with_scores(standalone_question, k=3)
    best_score = results[0][1]

    if best_score > threshold:
        answer = "I don't know based on the available company information."
        conversation_history.append({"user": question, "assistant": answer})

        return {
            "original_question": question,
            "standalone_question": standalone_question,
            "answer": answer,
            "sources": [],
            "confidence": "low",
            "best_score": best_score
        }

    relevant_docs = [doc for doc, score in results]
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    history_text = "\n".join([
        f"User: {turn['user']}\nAssistant: {turn['assistant']}"
        for turn in conversation_history[-4:]
    ])

    prompt = f"""
You are a helpful customer support assistant.

Use the conversation history to understand follow-up questions.
Answer using only the company information below.

Conversation history:
{history_text}

Company information:
{context}

Current user question:
{question}

If the answer is not in the company information, say:
"I don't know based on the available company information."

Answer:
"""

    response = call_llm_with_backoff(prompt)
    answer = response.text

    sources = [doc.metadata["source"] for doc in relevant_docs]

    conversation_history.append({"user": question, "assistant": answer})

    return {
        "original_question": question,
        "standalone_question": standalone_question,
        "answer": answer,
        "sources": list(set(sources)),
        "confidence": "high",
        "best_score": best_score
    }

Step 24: Rewrite follow up questions before retrieval

In [235]:
def rewrite_question_with_history(question):
    history_text = "\n".join([
        f"User: {turn['user']}\nAssistant: {turn['assistant']}"
        for turn in conversation_history[-4:]
    ])

    prompt = f"""
Rewrite the current user question as a complete standalone question.

Conversation history:
{history_text}

Current question:
{question}

Standalone question:
"""

    response = call_llm_with_backoff(prompt)
    return response.text

Step 26 : Re-test memory

In [236]:
conversation_history = []

result1 = support_agent_with_better_memory("Do you offer enterprise plans?")
print(result1["answer"])

result2 = support_agent_with_better_memory("What about SSO?")
print(result2["standalone_question"])
print(result2["answer"])

Yes, enterprise plans include priority support, custom integrations, and dedicated account management.
Do your enterprise plans include Single Sign-On (SSO)?
Yes, single sign-on (SSO) is available for enterprise customers using SAML or OAuth.


Step 27: Create a chat function:

This lets you test the agent like a real support chatbot.

It keeps running until the user types:

In [237]:
def chat():
    print("AI Customer Support Agent")
    print("Type 'exit' to stop.")
    print("-" * 50)

    while True:
        question = input("Customer: ")

        if question.lower() in ["exit", "quit", "stop"]:
            print("Chat ended.")
            break

        result = support_agent_with_better_memory(question)

        print("\nAssistant:")
        print(result["answer"])

        print("\nSources:", result["sources"])
        print("Confidence:", result["confidence"])
        print("-" * 50)

Step 28: Start the chatbot

In [238]:
conversation_history = []
chat()

AI Customer Support Agent
Type 'exit' to stop.
--------------------------------------------------

Assistant:
Yes, enterprise plans include priority support, custom integrations, and dedicated account management.

Sources: ['company_faq']
Confidence: high
--------------------------------------------------

Assistant:
Yes, single sign-on is available for enterprise customers using SAML or OAuth.

Sources: ['company_faq']
Confidence: high
--------------------------------------------------

Assistant:
I don't know based on the available company information.

Sources: ['company_faq']
Confidence: high
--------------------------------------------------

Assistant:
I don't know based on the available company information.

Sources: ['company_faq']
Confidence: high
--------------------------------------------------


KeyboardInterrupt: Interrupted by user